# 09 — Build Product Search Indexes

این نوت‌بوک Product Search V1 را آماده می‌کند:

1. canonicalize حدود 960k ردیف محصول به یک ردیف یکتا برای هر `product_id`
2. ساخت FAISS dense index روی metadata محصول
3. ساخت Tantivy BM25 global index روی metadata محصول

منطق اصلی در `src/rag/product_search/` قرار دارد.


In [ ]:
from pathlib import Path
import sys
import time

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.rag.config import load_config
from src.rag.preprocessing.processor import TextProcessor
from src.rag.embedding.factory import EmbeddingFactory
from src.rag.product_search import (
    build_canonical_products_file,
    ProductFAISSIndex,
    ProductBM25Index,
)


In [ ]:
PRODUCTS_PATH = PROJECT_ROOT / "data" / "processed" / "products_clean.parquet"
CANONICAL_PATH = PROJECT_ROOT / "data" / "processed" / "products_search.parquet"

DENSE_INDEX_PATH = PROJECT_ROOT / "data" / "indexes" / "products_embedding"
SPARSE_INDEX_PATH = PROJECT_ROOT / "data" / "indexes" / "products_bm25_tantivy"

RAG_CONFIG = load_config(PROJECT_ROOT / "configs" / "rag.yaml")
SEARCH_CONFIG = load_config(PROJECT_ROOT / "configs" / "product_search.yaml")

processor = TextProcessor()


## 1. Canonicalize products

In [ ]:
canonical = build_canonical_products_file(
    input_path=PRODUCTS_PATH,
    output_path=CANONICAL_PATH,
    overwrite=True,
)

print("Canonical products:", len(canonical))
print("Unique IDs:", canonical["id"].nunique())
display(canonical.head())


## 2. Build dense product index

In [ ]:
embedding_model = EmbeddingFactory.create(
    provider=RAG_CONFIG["embedding"]["provider"],
    model_name=RAG_CONFIG["embedding"]["model"],
)

index_cfg = SEARCH_CONFIG["indexing"]

start = time.perf_counter()

dense_manifest = ProductFAISSIndex.build_from_parquet(
    input_path=CANONICAL_PATH,
    output_path=DENSE_INDEX_PATH,
    embedding_model=embedding_model,
    processor=processor,
    chunk_size=index_cfg["dense_chunk_size"],
    encode_batch_size=index_cfg["dense_encode_batch_size"],
    overwrite=True,
)

print("Dense build minutes:", round((time.perf_counter()-start)/60, 2))
print(dense_manifest)


## 3. Build sparse product index

In [ ]:
start = time.perf_counter()

sparse_manifest = ProductBM25Index.build_from_parquet(
    input_path=CANONICAL_PATH,
    output_path=SPARSE_INDEX_PATH,
    processor=processor,
    batch_size=index_cfg["sparse_batch_size"],
    writer_heap_size=index_cfg["sparse_writer_heap_size"],
    num_threads=index_cfg["sparse_num_threads"],
    overwrite=True,
)

print("Sparse build minutes:", round((time.perf_counter()-start)/60, 2))
print(sparse_manifest)
